In [ ]:
import math
import warnings

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import matplotlib.pyplot as plt
import numpy as np
import sympy as sym

from rotation.symbolic.general import ANGLE, MASS, COUPLING, create_3d_rotation, create_4d_rotation
from rotation.symbolic.pseudo import create_mix_equation
from rotation.numerical.solver import MASS_NEUTRAL_PION, MASS_CHARGED_KAON, MASS_ETA, MASS_ETA_PRIME, G_PI_GAMMA, G_ETA_GAMMA, G_ETA_PRIME_GAMMA, initialize_system, create_func, solve_mixing
from rotation.numerical.pseudo import pseudo_calculate_meson_couplings, pseudo_calculate_bare_masses, pseudo_calculate_photon_coupling, pseudo_calculate_bare_kaon, pseudo_calculate_kaon_couplings

In [ ]:
def format_output(key, value):
    """Helps format solutions for printing"""
    if key in [MASS[k] for k in ["pi0", "pi0Bare"]]:
        value *= sym.Symbol(r" m_{\pi^0\text{, PDG}}") / MASS_NEUTRAL_PION
    elif key in [MASS[k] for k in ["eta", "etaBare"]]:
        value *= sym.Symbol(r" m_{\eta\text{, PDG}}") / MASS_ETA
    elif key in [MASS[k] for k in ["etaP", "etaPBare"]]:
        value *= sym.Symbol(r" m_{\eta^\prime\text{, PDG}}") / MASS_ETA_PRIME
    elif key in [MASS[k] for k in ["phi", "phiBare", "pi0-phi", "eta-phi"]]:
        value *= sym.Symbol(r" \text{ GeV}")
    elif key in ANGLE.values():
        value *= sym.Symbol(r" \pi") / math.pi
    return value

## Prepare solver

In [ ]:
rotate = create_4d_rotation()[0]
equation = create_mix_equation(rotate)[0]
argument_order, system_of_equations = initialize_system(equation.lhs - equation.rhs)

## Solver inputs

In [ ]:
# Fixed known parameters
mphi = 100
_x = 1
quarks = _x, -_x, -_x
bare_masses = pseudo_calculate_bare_masses(
    coupling_up=quarks[0],
    coupling_down=quarks[1],
    coupling_strange=quarks[2],
)
known = {
    MASS["pi0"]: MASS_NEUTRAL_PION,
    MASS["eta"]: MASS_ETA,
    MASS["phiBare"]: mphi,
    MASS["pi0-phi"]: bare_masses[MASS["pi0-phi"]],
    MASS["eta-phi"]: bare_masses[MASS["eta-phi"]],

    MASS["etaP"]: MASS_ETA_PRIME,
    MASS["etaP-phi"]: bare_masses[MASS["etaP-phi"]]
}

# These will be the unknowns we numerically solve for (the values are
# just initial guesses to get the solver going)
# Remember: the system of equations can only solve for up to 6 unknowns!
unknown = {
    MASS["phi"]: mphi,
    MASS["pi0Bare"]: MASS_NEUTRAL_PION,
    MASS["etaBare"]: MASS_ETA,
    ANGLE["alpha"]: 0.0,
    ANGLE["beta"]: 0.0,
    ANGLE["gamma"]: 0.0,

    MASS["etaPBare"]: MASS_ETA_PRIME,
    ANGLE["delta"]: 0.0,
    ANGLE["zeta"]: 0.0,
    ANGLE["kappa"]: 0.0,
}

# Printout the system with our known values, and unknowns blank
print("Arguments input:")
for k in argument_order:
    if k in known:
        sym.Eq(k, known[k], evaluate=False)
    else:
        sym.Eq(k, sym.Symbol("?"), evaluate=False)
equation.subs(known)

## Run solver

In [ ]:
# Create function
func = create_func(
    system_of_equations,
    argument_order,
    known,
)
try:
    # Solve system
    solved = solve_mixing(
        func,
        argument_order,
        unknown,
    )
    print("Solver succeeded!")
except ValueError as e:
    print(e)
    print("The solver failed! Try other inputs.")

## Solution display

In [ ]:
# Printout the system with our solved unknowns, and knowns blank
print("Arguments solved:")
for k, v in solved.items():
    v = format_output(k, v)
    sym.Eq(k, v, evaluate=False)

print("The solved equation:")
equation.subs(solved)

print("Compare LHS and RHS evaluations:")
sym.Eq(
    (1*equation.lhs.subs(known|solved)).evalf(),
    (1*equation.rhs.subs(known|solved)).evalf(),
    evaluate=False
)

## Calculate Couplings

In [ ]:
gchi = 0.2
meson_couplings = pseudo_calculate_meson_couplings(
    n_dim = rotate.rows,
    bare_mediator_dm_coupling = gchi,
    alpha = solved[ANGLE["alpha"]],
    beta = solved[ANGLE["beta"]],
    gamma = solved[ANGLE["gamma"]],
)
for s,g in meson_couplings.items():
    sym.Eq(s,g,evaluate=False)

In [ ]:
if ANGLE["delta"] not in solved:
    photon_coupling = pseudo_calculate_photon_coupling(
        n_dim = rotate.rows,
        alpha = solved[ANGLE["alpha"]],
        beta = solved[ANGLE["beta"]],
        gamma = solved[ANGLE["gamma"]],
    )
else:
    photon_coupling = pseudo_calculate_photon_coupling(
        n_dim = rotate.rows,
        alpha = solved[ANGLE["alpha"]],
        beta = solved[ANGLE["beta"]],
        gamma = solved[ANGLE["gamma"]],
        delta = solved[ANGLE["delta"]],
        zeta = solved[ANGLE["zeta"]],
        kappa = solved[ANGLE["kappa"]],
    )
sym.Eq(COUPLING["phi-gamma"], photon_coupling, evaluate=False)

In [ ]:
bare_kaon_couplings = pseudo_calculate_bare_kaon(energy_pion = 1, energy_kaon = 2)
for s,g in bare_kaon_couplings.items():
    sym.Eq(s,g,evaluate=False)

print("After rotating:")
if ANGLE["delta"] not in solved:
    kaon_couplings = pseudo_calculate_kaon_couplings(
        n_dim = rotate.rows,
        bare_pion_pion_kaon_coupling = bare_kaon_couplings[COUPLING["pi0Bare-pi-K"]],
        bare_mediator_pion_kaon_coupling = bare_kaon_couplings[COUPLING["phiBare-pi-K"]],
        bare_eta_pion_kaon_coupling = bare_kaon_couplings[COUPLING["etaBare-pi-K"]],
        bare_etaP_pion_kaon_coupling = bare_kaon_couplings[COUPLING["etaPBare-pi-K"]],
        alpha = solved[ANGLE["alpha"]],
        beta = solved[ANGLE["beta"]],
        gamma = solved[ANGLE["gamma"]],
    )
else:
    kaon_couplings = pseudo_calculate_kaon_couplings(
        n_dim = rotate.rows,
        bare_pion_pion_kaon_coupling = bare_kaon_couplings[COUPLING["pi0Bare-pi-K"]],
        bare_mediator_pion_kaon_coupling = bare_kaon_couplings[COUPLING["phiBare-pi-K"]],
        bare_eta_pion_kaon_coupling = bare_kaon_couplings[COUPLING["etaBare-pi-K"]],
        bare_etaP_pion_kaon_coupling = bare_kaon_couplings[COUPLING["etaPBare-pi-K"]],
        alpha = solved[ANGLE["alpha"]],
        beta = solved[ANGLE["beta"]],
        gamma = solved[ANGLE["gamma"]],
        delta = solved[ANGLE["delta"]],
        zeta = solved[ANGLE["zeta"]],
        kappa = solved[ANGLE["kappa"]],
    )
for s,g in kaon_couplings.items():
    sym.Eq(s,g,evaluate=False)

In [ ]:
# Creating the kaon couplings as a function of the pion energy
if ANGLE["delta"] not in solved:
    def kaon_function(energy):
        bare = pseudo_calculate_bare_kaon(energy_pion=energy, energy_kaon=2)
        return pseudo_calculate_kaon_couplings(
            n_dim=rotate.rows,
            bare_mediator_pion_kaon_coupling=bare[COUPLING[f"phiBare-pi-K"]],
            bare_pion_pion_kaon_coupling=bare[COUPLING[f"pi0Bare-pi-K"]],
            bare_eta_pion_kaon_coupling=bare[COUPLING[f"etaBare-pi-K"]],
            bare_etaP_pion_kaon_coupling=bare[COUPLING[f"etaPBare-pi-K"]],
            alpha=solved[ANGLE["alpha"]],
            beta=solved[ANGLE["beta"]],
            gamma=solved[ANGLE["gamma"]],
        )
else:
    def kaon_function(energy):
        bare = pseudo_calculate_bare_kaon(energy_pion=energy, energy_kaon=2)
        return pseudo_calculate_kaon_couplings(
            n_dim=rotate.rows,
            bare_mediator_pion_kaon_coupling=bare[COUPLING[f"phiBare-pi-K"]],
            bare_pion_pion_kaon_coupling=bare[COUPLING[f"pi0Bare-pi-K"]],
            bare_eta_pion_kaon_coupling=bare[COUPLING[f"etaBare-pi-K"]],
            bare_etaP_pion_kaon_coupling=bare[COUPLING[f"etaPBare-pi-K"]],
            alpha=solved[ANGLE["alpha"]],
            beta=solved[ANGLE["beta"]],
            gamma=solved[ANGLE["gamma"]],
            delta = solved[ANGLE["delta"]],
            zeta = solved[ANGLE["zeta"]],
            kappa = solved[ANGLE["kappa"]],
        )

xs = np.linspace(0,10,num=101)
fig, ax = plt.subplots(1,1)
for s, ys in kaon_function(xs).items():
    _ = ax.plot(xs, ys, label=f"${s}$")
_ = ax.set_xlabel("Pion Energy [GeV]")
_ = ax.legend()
_ = ax.grid()

## Loop the Solver

In [ ]:
pol = 1, 1, 1
# pol = 2, -1, -1
# pol = 1,-1, -1
x_label = f"\\text{{Quark coupling strength: (}}{',\\quad'.join([f'{COUPLING[s]}={p}' for s,p in zip('uds',pol)])} \\text{{)}}"
tol = 1e-13
m_phiBare = 100
x_min, x_max = 1e-1, 3.5

warnings.filterwarnings("error", category=RuntimeWarning)
solutions = []
for x in np.logspace(np.log10(x_min), np.log10(x_max), num=51):
    bare_masses = pseudo_calculate_bare_masses(
        coupling_up=pol[0]*x,
        coupling_down=pol[1]*x,
        coupling_strange=pol[2]*x,
    )
    known = {
        MASS["pi0"]: MASS_NEUTRAL_PION,
        MASS["eta"]: MASS_ETA,
        MASS["phiBare"]: m_phiBare,
        MASS["pi0-phi"]: bare_masses[MASS["pi0-phi"]],
        MASS["eta-phi"]: bare_masses[MASS["eta-phi"]],

        MASS["etaP"]: MASS_ETA_PRIME,
        MASS["etaP-phi"]: bare_masses[MASS["etaP-phi"]]
    }
    func = create_func(
        system_of_equations,
        argument_order,
        known,
    )
    try:
        solved = solve_mixing(
            func,
            argument_order,
            unknown,
            abs_tolerance=tol
        )
        solutions.append([x, known.copy(), solved.copy()])
    except ValueError as e:
        print(f"{x = :.3e}", end="\t")
        print(f"ValueError: {e}")
    except RuntimeWarning as e:
        print(f"{x = :.3e}", end="\t")
        print(f"RuntimeWarning: {e}")
warnings.resetwarnings()

In [ ]:
xs = np.array([sol[0] for sol in solutions])
# Reorganize solutions to be arrays of each unknown
data = {"in": {}, "out": {}}
for sol in solutions:
    for index, k in enumerate(["in","out"], start=1):
        for y_label, data_point in sol[index].items():
            try:
                data[k][y_label].append(data_point)
            except KeyError:
                data[k][y_label] = [data_point]
for k in data:
    for y_label, ys in data[k].items():
        data[k][y_label] = np.array(ys)
        data[k][y_label] = np.ma.array(ys, mask=data[k][y_label] < 0.0)

if not data["out"]:
    print("No data!")
else:
    g_chi = 1
    for k,v in pseudo_calculate_meson_couplings(
        n_dim = rotate.rows,
        bare_mediator_dm_coupling = g_chi,
        alpha = data["out"][ANGLE["alpha"]].data,
        beta = data["out"][ANGLE["beta"]].data,
        gamma = data["out"][ANGLE["gamma"]].data,
    ).items():
        data["out"][k] = np.ma.array(v)

    if ANGLE["delta"] not in data["out"]:
        data["out"][COUPLING["phi-gamma"]] = np.ma.array(pseudo_calculate_photon_coupling(
            n_dim = rotate.rows,
            alpha = data["out"][ANGLE["alpha"]].data,
            beta = data["out"][ANGLE["beta"]].data,
            gamma = data["out"][ANGLE["gamma"]].data,
        ))
    else:
        data["out"][COUPLING["phi-gamma"]] = np.ma.array(pseudo_calculate_photon_coupling(
            n_dim = rotate.rows,
            alpha = data["out"][ANGLE["alpha"]].data,
            beta = data["out"][ANGLE["beta"]].data,
            gamma = data["out"][ANGLE["gamma"]].data,
            delta = data["out"][ANGLE["delta"]].data,
            zeta = data["out"][ANGLE["zeta"]].data,
            kappa = data["out"][ANGLE["kappa"]].data,
        ))

In [ ]:
# plotting
m_phiBare = data["in"][MASS["phiBare"]].data
m_pi0_phi = data["in"][MASS["pi0-phi"]].data
m_eta_phi = data["in"][MASS["eta-phi"]].data
m_etaP_phi = data["in"][MASS["etaP-phi"]].data
alpha = data["out"][ANGLE["alpha"]].data
beta = data["out"][ANGLE["beta"]].data
gamma = data["out"][ANGLE["gamma"]].data
m_phi = data["out"][MASS["phi"]].data

small_mphi = (
    MASS_NEUTRAL_PION**2 * np.cos(beta) * np.cos(gamma) * np.sin(alpha),
    MASS_ETA * np.cos(gamma) * np.sin(beta),
    MASS_ETA_PRIME * np.sin(gamma),
)

small_approx = {
    ANGLE["alpha"]: 0.5 * m_pi0_phi**2/(MASS_NEUTRAL_PION**2 - m_phi**2),
    ANGLE["beta"]: 0.5 * m_eta_phi**2/(MASS_ETA**2 - m_phi**2),
    ANGLE["gamma"]: 0.5 * m_etaP_phi**2/(MASS_ETA_PRIME**2 - m_phi**2),
    MASS["phi"]: np.sqrt(m_phiBare**2 - small_mphi[0]**2 + small_mphi[1]**2 + small_mphi[2]**2) / np.abs(np.cos(alpha) * np.cos(beta) * np.cos(gamma)),
}
small_approx |= {
    COUPLING["pi0-chi"]: small_approx[ANGLE["alpha"]] * g_chi,
    COUPLING["eta-chi"]: small_approx[ANGLE["beta"]] * g_chi,
    COUPLING["etaP-chi"]: small_approx[ANGLE["gamma"]] * g_chi,
    COUPLING["phi-gamma"]: -G_PI_GAMMA*small_approx[ANGLE["alpha"]] - G_ETA_GAMMA*small_approx[ANGLE["beta"]] - G_ETA_PRIME_GAMMA*small_approx[ANGLE["gamma"]],
}

fig, axs = plt.subplots(2,3, figsize=(15,10), sharex=True)
mass_selector = {MASS[k]: v for k,v in zip(["pi0Bare", "etaBare", "etaPBare", "phi"], [MASS_NEUTRAL_PION, MASS_ETA, MASS_ETA_PRIME, mphi])}
colours = {
    r"\gamma}": "purple",
    r"\phi": "black",
    r"\pi_0": "blue",
    r"\eta^{\prime}": "green",
    r"\eta": "orange",
}
for y_label, ys in data["out"].items():
    style = None
    if r"_{b}" in str(y_label):
        style="--"
    colour = None
    for substring, c in colours.items():
        if substring in str(y_label):
            colour = c
            break
    if y_label in MASS.values():
        _ = axs[0,0].plot(xs, ys.data, label=f"${y_label}$", marker=".", linestyle=style, color=colour)
        ratio = mass_selector[y_label] / ys.data
        if y_label == MASS["phi"]:
            ratio = 1 / ratio
        _ = axs[1,0].plot(xs, ratio, label=f"${y_label}$", marker=".", linestyle=style, color=colour)
    elif y_label in COUPLING.values():
        _ = axs[0,1].plot(xs, ys.data, label=f"${y_label}$", marker=".", linestyle=style, color=colour)
        if y_label == COUPLING["phi-chi"]:
            ratio = g_chi / ys.data
        # elif y_label in [COUPLING[f"{k}-chi"] for k in ["pi0", "eta", "etaP"]]:
        #     ratio = 0 / ys.data # the bare couplings are zero by definition
        else:
            continue
        _ = axs[1,1].plot(xs, ratio, label=f"${y_label}$", marker=".", linestyle=style, color=colour)
    elif y_label in ANGLE.values():
        if y_label in [ANGLE[k] for k in ["delta", "zeta", "kappa"]]: # This will check if the current label is one of the delta, zeta, or kappa, and 'continue' skips this iteration of the plotting loop
            continue
        _ = axs[0,2].plot(xs, ys.data, label=f"${y_label}$", marker=".", linestyle=style, color=colour)
for y_label, approx in small_approx.items():
    if y_label in [ANGLE["alpha"], ANGLE["beta"], ANGLE["gamma"],]:
        continue
    if y_label in data["out"]:
        d = data["out"]
    elif y_label in data["in"]:
        d = data["in"]
    else:
        continue
    colour = None
    for substring, c in colours.items():
        if substring in str(y_label):
            colour = c
            break
    _ = axs[1,2].plot(xs, approx/d[y_label].data, label=f"${{{y_label}}}$", marker=".", color=colour)

_ = axs[0,0].set_ylabel("Mass [GeV]")
_ = axs[0,1].set_ylabel("Coupling")
_ = axs[0,2].set_ylabel("Angle")
_ = axs[1,0].set_ylabel("Bare Mass / Rotated Mass")
_ = axs[1,1].set_ylabel("Bare Coupling / Rotated Coupling")
_ = axs[1,2].set_ylabel("Small Approx / Calculator Solved")
# _ = fig.suptitle(", ".join([f"${COUPLING[q]} = {quarks[i]}$" for i,q in enumerate("uds")]))
_ = fig.suptitle(f"${{{MASS['phiBare']}}}={{{mphi}}}$ GeV, $|\\text{{tolerance}}| = {{{tol:.1e}}}$, ${{{COUPLING['chi']}}}=${g_chi} GeV")
for ax in axs[-1,:]:
    _ = ax.set_xlabel(f"${x_label}$")
for ax in axs.flatten():
    _ = ax.set_xlim(x_min, x_max)
    _ = ax.legend()
    _ = ax.grid(True)
_ = fig.tight_layout()
_ = axs[0,0].set_xscale("log")
_ = axs[0,0].set_yscale("log")
_ = axs[0,1].set_yscale("symlog", linthresh=1e-10)
_ = axs[1,0].set_yscale("log")
_ = axs[1,2].set_yscale("log")
fig.show()

In [ ]:
# ratio vs fractional error: ratio better for extreme differences, fractional better for smaller
# Plot sets for: Heavy, Light, and one near a meson mass

def angle_plot(ax, xs, alpha, beta, gamma, delta=None, zeta=None, kappa=None):
    angles = {
        ANGLE["alpha"]: alpha,
        ANGLE["beta"]: beta,
        ANGLE["gamma"]: gamma,
        ANGLE["delta"]: delta,
        ANGLE["zeta"]: zeta,
        ANGLE["kappa"]: kappa,
    }
    for k,c in zip([ANGLE["alpha"], ANGLE["beta"], ANGLE["gamma"]], plt.get_cmap("tab10").colors[:3]):
        if angles[k] is None:
            continue
        ax.plot(xs, angles[k], color=c, label=f"${k}$")
    ylim = ax.get_ylim()
    for k,c in zip([ANGLE["delta"], ANGLE["zeta"], ANGLE["kappa"]], plt.get_cmap("tab10").colors[3:6]):
        if angles[k] is None:
            continue
        ax.plot(xs, angles[k], color=c, label=f"${k}$", linestyle="-.")
    ax.set_ylim(ylim)
    ax.set_ylabel("Angle [Rad]")

    sec_y = ax.secondary_yaxis("right", functions=(lambda x: x*180/np.pi, lambda x: x*np.pi/180))
    sec_y.set_ylabel("Angle [Deg]")
    return sec_y

def mass_plot(ax, xs, phi_bare=None, phi=None, pi0_bare=None, pi0=None, eta_bare=None, eta=None, etaP_bare=None, etaP=None):
    masses = {
        "phi": (phi_bare, phi),
        "pi0": (pi0_bare, pi0),
        "eta": (eta_bare, eta),
        "etaP": (etaP_bare, etaP),
    }
    ax_sec = ax.twinx()
    colours = np.array(plt.get_cmap("tab20").colors)
    for (k,(bare,rotated)), c in zip(masses.items(), colours.reshape(-1,2,3)):
        col = c
        if k == "phi":
            col = col[::-1]
        if rotated is not None:
            ax.plot(xs, rotated, label=f"${MASS[k]}$", linestyle="-.", color=col[1])
        if bare is not None:
            ax.plot(xs, bare, label=f"${MASS[k+'Bare']}$", linestyle="--", color=col[0])
        if bare is not None and rotated is not None:
            custom_linestyle = {"label": f"${MASS[k]}$/${MASS[k+'Bare']}$", "linestyle": ":", "color": c[0]}
            ax_sec.plot(xs, rotated/bare, **custom_linestyle)
            ax.plot([], [], **custom_linestyle) # dummy plot to make the label appear on the original axis
    ax.set_ylabel("Mass [GeV]")
    ax_sec.set_ylabel("Rotated to Bare Ratio")
    return ax_sec

# All rotated meson couplings / small angle meson couplings (note the bare phi-chi coupling set by user?)
def meson_plot(ax):
    return None

# All photon couplings / small angle photon couplings


# Kaon couplings / small angle kaon couplings (only if is easy)

fig, axs = plt.subplots(3,1,squeeze=False, figsize=(6,12))
# angle_plot(axs[0,0], xs, *[data["out"][ANGLE[k]].data for k in ["alpha", "beta", "gamma"]])
ax_degree = angle_plot(axs[0,0], xs, *[data["out"][ANGLE[k]].data for k in ["alpha", "beta", "gamma", "delta", "zeta", "kappa"]])
ax_ratio = mass_plot(axs[1,0], xs, phi_bare=data["in"][MASS["phiBare"]], phi=data["out"][MASS["phi"]], eta_bare=data["out"][MASS["etaBare"]], eta=data["in"][MASS["eta"]], etaP_bare=data["out"][MASS["etaPBare"]], etaP=data["in"][MASS["etaP"]])
meson_plot(axs[2,0])
for ax in axs.flat:
    _ = ax.legend()
fig.tight_layout()